# PuttingCupintotheDish Demo Visualization

Explore demo dataset structure and visualize videos from `/media/hyunjin/T7/rby1_demo/PuttingCupintotheDishV2`.

In [ ]:
from pathlib import Path

# ═══════════════════════════════════════════════════════
#  Global Settings — edit here before running all cells
# ═══════════════════════════════════════════════════════
EPISODE_IDX        = 90    # Default episode index used across sections
DATASET_ROOT       = Path("/media/hyunjin/T7/rby1_demo/PuttingCupintotheDishV2")


In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.animation as animation
from pathlib import Path
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['axes.unicode_minus'] = False

print(f"Dataset root  : {DATASET_ROOT}")
print(f"Dataset exists: {DATASET_ROOT.exists()}")


## 1. Explore Full Dataset Structure

In [ ]:
# Collect episode directories
episode_dirs = sorted(DATASET_ROOT.glob("episode_*"), key=lambda x: int(x.name.split("_")[1]))
print(f"Total episodes: {len(episode_dirs)}")
print(f"Episode list: {[d.name for d in episode_dirs]}\n")

# Summarize files and HDF5 structure for each episode
summary = []
for ep_dir in episode_dirs:
    files = list(ep_dir.iterdir())
    h5_files = [f for f in files if f.suffix == '.h5']
    other_files = [f for f in files if f.suffix != '.h5']
    
    info = {"episode": ep_dir.name, "h5_files": [f.name for f in h5_files], "other": [f.name for f in other_files],
            "n_steps": 0, "n_frames": 0, "groups": []}
    
    # Read metadata from HDF5 file
    if h5_files:
        try:
            with h5py.File(h5_files[0], 'r') as f:
                info["groups"] = list(f.keys())
                info["n_steps"]  = f['samples/time'].shape[0] if 'samples/time' in f else 0
                info["n_frames"] = f['head_rgb/image'].shape[0] if 'head_rgb/image' in f else 0
        except Exception as e:
            print(f"  [WARN] Failed to read {h5_files[0].name}: {e}")
    
    summary.append(info)
    print(f"[{ep_dir.name}] n_steps={info['n_steps']}, n_frames(rgb)={info['n_frames']}, files={[f.name for f in files]}")

In [ ]:
# Visualize n_steps and n_frames per episode
n_steps_list = [s['n_steps'] for s in summary]
n_frames_list = [s['n_frames'] for s in summary]
ep_indices = [int(s['episode'].split('_')[1]) for s in summary]

title_steps = "Robot State Steps per Episode"
title_frames = "RGB Frames per Episode (head_rgb)"
mean_label = "Mean"
overview_title = f"PuttingCupintotheDish Dataset Overview ({len(summary)} episodes)"
steps_stats_label = "[Steps Stats]"
frames_stats_label = "[Frames Stats]"

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(ep_indices, n_steps_list, color='steelblue', alpha=0.8)
axes[0].set_title(title_steps, fontsize=13)
axes[0].set_xlabel("Episode Index")
axes[0].set_ylabel("Timesteps")
axes[0].axhline(np.mean(n_steps_list), color='red', linestyle='--', label=f"{mean_label}: {np.mean(n_steps_list):.1f}")
axes[0].legend()

axes[1].bar(ep_indices, n_frames_list, color='coral', alpha=0.8)
axes[1].set_title(title_frames, fontsize=13)
axes[1].set_xlabel("Episode Index")
axes[1].set_ylabel("Frames")
axes[1].axhline(np.mean(n_frames_list), color='red', linestyle='--', label=f"{mean_label}: {np.mean(n_frames_list):.1f}")
axes[1].legend()

plt.tight_layout()
plt.suptitle(overview_title, y=1.02, fontsize=14, fontweight='bold')
plt.show()

print(f"\n{steps_stats_label} min={min(n_steps_list)}, max={max(n_steps_list)}, mean={np.mean(n_steps_list):.1f}")
print(f"{frames_stats_label} min={min(n_frames_list)}, max={max(n_frames_list)}, mean={np.mean(n_frames_list):.1f}")

## 2. Inspect Single Episode HDF5 Structure

In [ ]:
# EPISODE_IDX is set in the Settings cell (cell 1)
ep_dir = DATASET_ROOT / f"episode_{EPISODE_IDX}"
h5_path = list(ep_dir.glob("*.h5"))[0]
print(f"File: {h5_path}\n")

def print_h5_structure(name, obj):
    indent = "  " * name.count("/")
    if isinstance(obj, h5py.Dataset):
        print(f"{indent}📊 [{name}]  shape={obj.shape}  dtype={obj.dtype}")
    elif isinstance(obj, h5py.Group):
        print(f"{indent}📁 [{name}]")

with h5py.File(h5_path, 'r') as f:
    print("=== HDF5 File Structure ===")
    f.visititems(print_h5_structure)
    
    print("\n=== samples keys ===")
    if 'samples' in f:
        for key in f['samples'].keys():
            ds = f[f'samples/{key}']
            print(f"  samples/{key}: shape={ds.shape}, dtype={ds.dtype}")

In [ ]:
# Compare sample and image timelines across one or multiple episodes
CHECK_EPISODES = "all"  # e.g., "all" or [0, 1, 2, 10]

def analyze_episode_timeline(ep_idx):
    ep_dir_local = DATASET_ROOT / f"episode_{ep_idx}"
    h5_candidates = list(ep_dir_local.glob("*.h5"))
    if not h5_candidates:
        print(f"[episode_{ep_idx}] no .h5 file found")
        return None

    h5_local = h5_candidates[0]
    with h5py.File(h5_local, 'r') as f:
        if 'samples/time' not in f:
            print(f"[episode_{ep_idx}] samples/time not found")
            return None

        sample_t = f['samples/time'][:]
        head_rgb_t = f['head_rgb/time'][:] if 'head_rgb/time' in f else None

    row = {
        'episode': ep_idx,
        'h5': h5_local.name,
        'samples_len': len(sample_t),
        'head_rgb_len': len(head_rgb_t) if head_rgb_t is not None else 0,
        'len_diff': (len(sample_t) - len(head_rgb_t)) if head_rgb_t is not None else np.nan,
        'samples_dur': float(sample_t[-1] - sample_t[0]) if len(sample_t) > 1 else 0.0,
        'head_rgb_dur': float(head_rgb_t[-1] - head_rgb_t[0]) if (head_rgb_t is not None and len(head_rgb_t) > 1) else np.nan,
        'samples_hz': float(1 / np.diff(sample_t).mean()) if len(sample_t) > 1 else np.nan,
        'head_rgb_hz': float(1 / np.diff(head_rgb_t).mean()) if (head_rgb_t is not None and len(head_rgb_t) > 1) else np.nan,
        'outside_count': int(np.sum((sample_t < head_rgb_t[0]) | (sample_t > head_rgb_t[-1]))) if (head_rgb_t is not None and len(head_rgb_t) > 0) else np.nan,
    }
    return row

# Resolve episode list
if CHECK_EPISODES == "all":
    check_indices = [int(d.name.split('_')[1]) for d in episode_dirs]
else:
    check_indices = CHECK_EPISODES

timeline_rows = []
for ep_idx in check_indices:
    result = analyze_episode_timeline(ep_idx)
    if result is not None:
        timeline_rows.append(result)

if timeline_rows:
    print("episode | samples | head_rgb | diff | samples_hz | head_rgb_hz | outside")
    print("-" * 78)
    for r in sorted(timeline_rows, key=lambda x: x['episode']):
        print(
            f"{r['episode']:>7} | {r['samples_len']:>7} | {r['head_rgb_len']:>8} | {r['len_diff']:>4} | "
            f"{r['samples_hz']:>9.2f} | {r['head_rgb_hz']:>11.2f} | {r['outside_count']}"
        )

    diffs = np.array([r['len_diff'] for r in timeline_rows if not np.isnan(r['len_diff'])])
    print("\nSummary:")
    print(f"checked episodes: {len(timeline_rows)}")
    print(f"len diff min/max/mean: {diffs.min():.0f} / {diffs.max():.0f} / {diffs.mean():.2f}")
else:
    print("No valid episodes were analyzed.")

## 3. Visualize RGB Camera Frames (head / left / right)

In [ ]:
N_SAMPLE_FRAMES = 6  # Number of frames to visualize

cameras = ['head_rgb', 'left_rgb', 'right_rgb']
cam_labels = {'head_rgb': 'Head Camera', 'left_rgb': 'Left Camera', 'right_rgb': 'Right Camera'}

with h5py.File(h5_path, 'r') as f:
    # Sample N_SAMPLE_FRAMES frames uniformly from each camera
    cam_frames = {}
    for cam in cameras:
        if f'{cam}/image' in f:
            images = f[f'{cam}/image'][:]  # (T, H, W, 3)
            total = images.shape[0]
            indices = np.linspace(0, total - 1, N_SAMPLE_FRAMES, dtype=int)
            cam_frames[cam] = (images[indices], indices, total)

fig, axes = plt.subplots(len(cam_frames), N_SAMPLE_FRAMES, figsize=(N_SAMPLE_FRAMES * 3, len(cam_frames) * 2.5))
fig.suptitle(f"Episode {EPISODE_IDX} — RGB Camera Frames", fontsize=15, fontweight='bold')

for row, cam in enumerate(cameras):
    if cam not in cam_frames:
        continue
    images, indices, total = cam_frames[cam]
    for col in range(N_SAMPLE_FRAMES):
        ax = axes[row][col]
        ax.imshow(images[col])
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(cam_labels[cam], fontsize=11, rotation=90, labelpad=10)
            ax.yaxis.set_label_coords(-0.15, 0.5)
        ax.set_title(f"t={indices[col]}/{total-1}", fontsize=8)

plt.tight_layout()
plt.show()

## 4. Visualize Depth Images (head_depth)

In [ ]:
with h5py.File(h5_path, 'r') as f:
    if 'head_depth/image' in f:
        depth_images = f['head_depth/image'][:]  # (T, H, W) uint16
        total = depth_images.shape[0]
        indices = np.linspace(0, total - 1, N_SAMPLE_FRAMES, dtype=int)
        sampled_depth = depth_images[indices]
    else:
        sampled_depth = None
        print("No head_depth/image data found")

if sampled_depth is not None:
    fig, axes = plt.subplots(1, N_SAMPLE_FRAMES, figsize=(N_SAMPLE_FRAMES * 3, 2.8))
    fig.suptitle(f"Episode {EPISODE_IDX} — Head Depth Frames", fontsize=14, fontweight='bold')
    
    for col in range(N_SAMPLE_FRAMES):
        ax = axes[col]
        depth_frame = sampled_depth[col].astype(np.float32)
        # Clip to valid depth range (0~5000 mm)
        depth_frame = np.clip(depth_frame, 0, 5000)
        im = ax.imshow(depth_frame, cmap='plasma', vmin=0, vmax=5000)
        ax.axis('off')
        ax.set_title(f"t={indices[col]}", fontsize=9)
    
    plt.colorbar(im, ax=axes[-1], label="Depth (mm)", shrink=0.8)
    plt.tight_layout()
    plt.show()
    print(f"Depth stats: min={depth_images.min()}, max={depth_images.max()}, mean={depth_images.mean():.1f}")

## 5. Robot State & Action — Time-Series Visualization

`robot_position` (현재 관절 상태) vs `robot_target_joints` (액션/목표 관절) 및  
`gripper_state` vs `gripper_target` 을 **Body Part 별**로 비교합니다.

| Joint indices | Group |
|---|---|
| J0 – J5   | Torso (6 joints) |
| J6 – J12  | Right Arm (7 joints) |
| J13 – J19 | Left Arm (7 joints) |
| J20 – J23 | Head / Other (4 joints) |

**Figure 1** Overview — 각 그룹 state(실선) vs action(점선) 오버뷰  
**Figure 2** Right Arm Detail — J6~J12 개별 서브플롯 + RMSE  
**Figure 3** Left Arm Detail — J13~J19 개별 서브플롯 + RMSE  
**Figure 4** Gripper Detail — Left / Right 그리퍼 state vs action  
**Figure 5** Torso & Head Detail — J0~J5, J20~J23 개별 서브플롯


In [ ]:
# ──────────────────────────────────────────────
#  User options
# ──────────────────────────────────────────────
TS_EPISODE_IDX = EPISODE_IDX   # episode to inspect
DEGREES        = True           # True → degrees, False → radians

# ──────────────────────────────────────────────
#  Joint group definitions
# ──────────────────────────────────────────────
# Joint index mapping for RBY1 (24 DOF total)
# J0–J5   : Torso  (6 joints)
# J6–J12  : Right Arm (7 joints)
# J13–J19 : Left Arm  (7 joints)
# J20–J23 : Head + Other (4 joints)
JOINT_GROUPS = [
    ("Torso",     list(range(0,  6)),  "tab:blue"),
    ("Right Arm", list(range(6,  13)), "tab:orange"),
    ("Left Arm",  list(range(13, 20)), "tab:green"),
    ("Head",      list(range(20, 24)), "tab:purple"),
]

def joint_color(j):
    for _, idxs, col in JOINT_GROUPS:
        if j in idxs:
            return col
    return "gray"

def joint_label(j):
    prefixes = {0: "T", 6: "R", 13: "L", 20: "H"}
    for _, idxs, _ in JOINT_GROUPS:
        if j in idxs:
            start = idxs[0]
            return f"{prefixes.get(start, 'J')}{j - start}"
    return f"J{j}"

# ──────────────────────────────────────────────
#  Load data
# ──────────────────────────────────────────────
ep_dir_ts  = DATASET_ROOT / f"episode_{TS_EPISODE_IDX}"
h5_path_ts = list(ep_dir_ts.glob("*.h5"))[0]

with h5py.File(h5_path_ts, "r") as f:
    time_ts          = f["samples/time"][:]
    robot_pos_ts     = f["samples/robot_position"][:]        # (T, 24)
    robot_tgt_ts     = f["samples/robot_target_joints"][:]   # (T, 24)
    gripper_state_ts = f["samples/gripper_state"][:]         # (T,  2)  [left, right]
    gripper_tgt_ts   = f["samples/gripper_target"][:]        # (T,  2)
    base_state_ts    = f["samples/base_state"][:]            # (T,  3)

time_rel_ts = time_ts - time_ts[0]
n_jt        = robot_pos_ts.shape[1]   # 24
T_end       = time_rel_ts[-1]

# Unit conversion
_sc   = np.degrees if DEGREES else (lambda x: x)
_unit = "°" if DEGREES else "rad"
#  Figure 1 ─ Overview by group  (4 joint groups + gripper/base row)
rt    = _sc(robot_tgt_ts)    # action
fig1 = plt.figure(figsize=(16, 16))
gs1  = gridspec.GridSpec(5, 3, figure=fig1, hspace=0.55, wspace=0.30)

group_axes = []
for gi, (grp, idxs, col) in enumerate(JOINT_GROUPS):
    ax = fig1.add_subplot(gs1[gi, :])   # full-width row
    for j in idxs:
        ax.plot(time_rel_ts, rp[:, j], color=col, lw=1.1, alpha=0.75)
        ax.plot(time_rel_ts, rt[:, j], color=col, lw=0.9, alpha=0.35, ls="--")
    ax.set_xlim(0, T_end)
    ax.set_title(f"{grp}  (J{idxs[0]}–J{idxs[-1]})  —  solid=State  dashed=Action",
                 fontsize=10, fontweight="bold")
    ax.set_ylabel(f"Angle ({_unit})", fontsize=8)
    ax.grid(True, alpha=0.2)
    group_axes.append(ax)

# Row 4 : Left Gripper | Right Gripper | Base
ax_lg = fig1.add_subplot(gs1[4, 0])
ax_rg = fig1.add_subplot(gs1[4, 1])
ax_bs = fig1.add_subplot(gs1[4, 2])

for ax_g, k, name, sc, ac in [
    (ax_lg, 1, "Left Gripper",  "royalblue", "steelblue"),  # data[1]=left
    (ax_rg, 0, "Right Gripper", "tomato",    "firebrick"),  # data[0]=right
]:
    ax_g.plot(time_rel_ts, gripper_state_ts[:, k], color=sc, lw=2.0, label="State")
    ax_g.plot(time_rel_ts, gripper_tgt_ts[:,   k], color=ac, lw=1.6, ls="--", alpha=0.7, label="Action")
    ax_g.fill_between(time_rel_ts, gripper_state_ts[:, k], gripper_tgt_ts[:, k],
                      alpha=0.15, color=sc)
    rmse_g = float(np.sqrt(np.mean((gripper_state_ts[:, k] - gripper_tgt_ts[:, k])**2)))
    ax_g.set_title(f"{name}  RMSE={rmse_g:.4f}", fontsize=10, fontweight="bold")
    ax_g.set_xlabel("Time (s)", fontsize=8)
    ax_g.set_ylabel("Value", fontsize=8)
    ax_g.legend(fontsize=8)
    ax_g.grid(True, alpha=0.2)
    ax_g.set_xlim(0, T_end)

_bc = ["steelblue", "coral", "seagreen"]
_bl = ["x (m)", "y (m)", "yaw (rad)"]
for i, (lbl, col) in enumerate(zip(_bl, _bc)):
    ax_bs.plot(time_rel_ts, base_state_ts[:, i], color=col, lw=1.8, label=lbl)
ax_bs.set_title("Base State", fontsize=10, fontweight="bold")
ax_bs.set_xlabel("Time (s)", fontsize=8)
ax_bs.legend(fontsize=8)
ax_bs.grid(True, alpha=0.2)
ax_bs.set_xlim(0, T_end)

fig1.suptitle(
    f"Episode {TS_EPISODE_IDX}  —  State (solid) vs Action (dashed)  |  Overview by Group",
    fontsize=14, fontweight="bold", y=1.005,
)
plt.show()

# ──────────────────────────────────────────────────────────────────
#  Figure 2 ─ Right Arm Detail  (J6–J12, 7 joints)
    f"Episode {TS_EPISODE_IDX}  —  State (solid) vs Action (dashed)  |  Overview by Group",
right_idxs = JOINT_GROUPS[1][1]   # J6-J12 (Right Arm)
fig2, axes2 = plt.subplots(2, 4, figsize=(16, 7), constrained_layout=True)
right_idxs = JOINT_GROUPS[1][1]   # J6–J12 (Right Arm)
    f"Episode {TS_EPISODE_IDX}  —  Right Arm  (J6–J12)  State vs Action",
    fontsize=13, fontweight="bold"
)
axes2_flat = axes2.flatten()
for pi, j in enumerate(right_idxs):
    ax = axes2_flat[pi]
    lbl = joint_label(j)
    ax.plot(time_rel_ts, rp[:, j], color="steelblue", lw=1.4, label="State")
    ax.plot(time_rel_ts, rt[:, j], color="darkorange", lw=1.3, ls="--", label="Action")
    ax.fill_between(time_rel_ts, rp[:, j], rt[:, j], alpha=0.15, color="tab:orange")
    rmse_j = float(np.sqrt(np.mean(err[:, j]**2)))
    ax.set_title(f"J{j} [{lbl}]  RMSE={rmse_j:.2f}{_unit}", fontsize=9, fontweight="bold")
    ax.set_xlabel("Time (s)", fontsize=7)
    ax.set_ylabel(_unit, fontsize=7)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.2)
    if pi == 0:
        ax.legend(fontsize=7)
    ax.set_xlim(0, T_end)
axes2_flat[-1].set_visible(False)   # 8th slot empty
plt.show()

# ──────────────────────────────────────────────────────────────────
#  Figure 3 ─ Left Arm Detail  (J13–J19, 7 joints)
        ax.legend(fontsize=7)
left_idxs = JOINT_GROUPS[2][1]   # J13-J19 (Left Arm)
fig3, axes3 = plt.subplots(2, 4, figsize=(16, 7), constrained_layout=True)
left_idxs = JOINT_GROUPS[2][1]   # J13–J19 (Left Arm)
    f"Episode {TS_EPISODE_IDX}  —  Left Arm  (J13–J19)  State vs Action",
    fontsize=13, fontweight="bold"
)
axes3_flat = axes3.flatten()
for pi, j in enumerate(left_idxs):
    ax = axes3_flat[pi]
    lbl = joint_label(j)
    ax.plot(time_rel_ts, rp[:, j], color="steelblue", lw=1.4, label="State")
    ax.plot(time_rel_ts, rt[:, j], color="seagreen",  lw=1.3, ls="--", label="Action")
    ax.fill_between(time_rel_ts, rp[:, j], rt[:, j], alpha=0.15, color="tab:green")
    rmse_j = float(np.sqrt(np.mean(err[:, j]**2)))
    ax.set_title(f"J{j} [{lbl}]  RMSE={rmse_j:.2f}{_unit}", fontsize=9, fontweight="bold")
    ax.set_xlabel("Time (s)", fontsize=7)
    ax.set_ylabel(_unit, fontsize=7)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.2)
    if pi == 0:
        ax.legend(fontsize=7)
    ax.set_xlim(0, T_end)
axes3_flat[-1].set_visible(False)
plt.show()

# ──────────────────────────────────────────────────────────────────
#  Figure 4 ─ Gripper Detail  (Left + Right)
        ax.legend(fontsize=7)
fig4, axes4 = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
fig4.suptitle(
    f"Episode {TS_EPISODE_IDX}  —  Gripper Detail  State vs Action",
    fontsize=13, fontweight="bold"
)
for k, (name, sc, ac) in enumerate([
    ("Left Gripper",  "royalblue", "steelblue"),
    ("Right Gripper", "tomato",    "firebrick"),
]):
    ax = axes4[k]
    s_g = gripper_state_ts[:, 1 - k]  # data: [right,left]; display: [Left,Right]
    a_g = gripper_tgt_ts[:, 1 - k]
    ax.plot(time_rel_ts, s_g, color=sc, lw=2.2, label="State")
    ax.plot(time_rel_ts, a_g, color=ac, lw=1.8, ls="--", label="Action")
    ax.fill_between(time_rel_ts, s_g, a_g, alpha=0.18, color=sc)
    rmse_g = float(np.sqrt(np.mean((s_g - a_g)**2)))
    ax.set_title(f"{name}  RMSE={rmse_g:.5f}", fontsize=11, fontweight="bold")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Value")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.25)
    ax.set_xlim(0, T_end)
plt.show()

# ──────────────────────────────────────────────────────────────────
#  Figure 5 ─ Torso & Other Detail
    ax.legend(fontsize=9)
fig5, axes5 = plt.subplots(2, 6, figsize=(18, 6), constrained_layout=True)
fig5.suptitle(
    f"Episode {TS_EPISODE_IDX}  —  Torso (J0–J5) & Head (J20–J23)  State vs Action",
    fontsize=13, fontweight="bold"
    f"Episode {TS_EPISODE_IDX}  —  Torso (J0–J5) & Head/Other (J20–J23)  State vs Action",
torso_other_idxs = JOINT_GROUPS[0][1] + JOINT_GROUPS[3][1]  # Torso + Head
axes5_flat = axes5.flatten()
torso_other_idxs = JOINT_GROUPS[0][1] + JOINT_GROUPS[3][1]  # Torso + Head
    ax = axes5_flat[pi]
    lbl = joint_label(j)
    col_fill = joint_color(j)
    ax.plot(time_rel_ts, rp[:, j], color="steelblue",  lw=1.4, label="State")
    ax.plot(time_rel_ts, rt[:, j], color="darkorange",  lw=1.3, ls="--", label="Action")
    ax.fill_between(time_rel_ts, rp[:, j], rt[:, j], alpha=0.15, color=col_fill)
    rmse_j = float(np.sqrt(np.mean(err[:, j]**2)))
    ax.set_title(f"J{j} [{lbl}]  RMSE={rmse_j:.2f}{_unit}", fontsize=8.5, fontweight="bold")
    ax.set_xlabel("Time (s)", fontsize=7)
    ax.set_ylabel(_unit, fontsize=7)
    ax.tick_params(labelsize=7)
    ax.grid(True, alpha=0.2)
    if pi == 0:
        ax.legend(fontsize=7)
    ax.set_xlim(0, T_end)
# hide unused (12 slots, 10 joints)
for pi in range(len(torso_other_idxs), len(axes5_flat)):
    axes5_flat[pi].set_visible(False)
plt.show()

# ──────────────────────────────────────────────────────────────────
#  Summary Statistics
# hide unused (12 slots, 10 joints)
abs_err = np.abs(err)
rmse_all = np.sqrt(np.mean(err**2, axis=0))

print(f"\n{'─'*72}")
print(f"Episode {TS_EPISODE_IDX}  |  T={len(time_rel_ts)} steps  |  "
      f"duration={T_end:.2f}s  |  unit={_unit}")
print(f"{'─'*72}")
print(f"{'Joint':<8} {'Group':<11} {'Mean|err|':>10} {'Max|err|':>10} {'RMSE':>9}")
rmse_all = np.sqrt(np.mean(err**2, axis=0))
for j in range(n_jt):
    grp_name = next((g for g, idxs, _ in JOINT_GROUPS if j in idxs), "Unknown")
    print(f"  J{j:<5} {grp_name:<11} "
    grp_name = next((g for g, idxs, _ in JOINT_GROUPS if j in idxs), "Unknown")
    print(f"  J{j:<5} {grp_name:<12} "
          f"{rmse_all[j]:>9.3f}")
print(f"{'─'*72}")
print(f"{'Overall':>20}   RMSE = {float(np.sqrt(np.mean(err**2))):.4f} {_unit}")

g_err = gripper_state_ts - gripper_tgt_ts
print(f"\nGripper  Left  RMSE={np.sqrt(np.mean(g_err[:,1]**2)):.5f}  "
      f"Right RMSE={np.sqrt(np.mean(g_err[:,0]**2)):.5f}")

print(f"{'─'*72}")


g_err = gripper_state_ts - gripper_tgt_ts      f"Right RMSE={np.sqrt(np.mean(g_err[:,0]**2)):.5f}")
print(f"\nGripper  Left  RMSE={np.sqrt(np.mean(g_err[:,1]**2)):.5f}  "

## 6. Create and Play Video (head / left / right RGB)

In [ ]:
VIDEO_EPISODE_IDX = 70  # Episode index for video generation (change as needed)
VIDEO_CAMERA = 'head_rgb'  # Choose from: 'head_rgb', 'left_rgb', 'right_rgb'
VIDEO_FPS = 15             # Playback FPS

ep_dir_v = DATASET_ROOT / f"episode_{VIDEO_EPISODE_IDX}"
h5_path_v = list(ep_dir_v.glob("*.h5"))[0]

with h5py.File(h5_path_v, 'r') as f:
    frames = f[f'{VIDEO_CAMERA}/image'][:]

print(f"Episode {VIDEO_EPISODE_IDX}, camera: {VIDEO_CAMERA}")
print(f"Total {frames.shape[0]} frames, resolution: {frames.shape[2]}x{frames.shape[1]}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.axis('off')
ax.set_title(f"Episode {VIDEO_EPISODE_IDX} — {VIDEO_CAMERA}", fontsize=12, fontweight='bold')

im = ax.imshow(frames[0])
frame_text = ax.text(5, 15, '', color='white', fontsize=11, fontweight='bold',
                     bbox=dict(facecolor='black', alpha=0.5, boxstyle='round,pad=0.2'))

def update(i):
    im.set_data(frames[i])
    frame_text.set_text(f"Frame {i}/{len(frames)-1}")
    return [im, frame_text]

ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=1000 // VIDEO_FPS, blit=True)
plt.close()

HTML(ani.to_jshtml())

## 7. Synchronized 3-Camera Playback (head / left / right)

In [ ]:
MULTI_EPISODE_IDX = 0  # Episode index (change as needed)
MULTI_FPS = 10

ep_dir_m = DATASET_ROOT / f"episode_{MULTI_EPISODE_IDX}"
h5_path_m = list(ep_dir_m.glob("*.h5"))[0]

with h5py.File(h5_path_m, 'r') as f:
    all_frames = {}
    for cam in ['head_rgb', 'left_rgb', 'right_rgb']:
        if f'{cam}/image' in f:
            all_frames[cam] = f[f'{cam}/image'][:]
        else:
            all_frames[cam] = None

# Keep only available cameras (all three)
valid_cams = [(cam, all_frames[cam]) for cam in ['head_rgb', 'left_rgb', 'right_rgb'] if all_frames[cam] is not None]
n_cams = len(valid_cams)
# Minimum shared frame count across cameras
min_frames = min(frms.shape[0] for _, frms in valid_cams)

print(f"Episode {MULTI_EPISODE_IDX}: {n_cams} cameras, minimum shared frames={min_frames}")

fig, axes = plt.subplots(1, n_cams, figsize=(7 * n_cams, 4.5))
if n_cams == 1:
    axes = [axes]

ims = []
texts = []
for ax, (cam, frms) in zip(axes, valid_cams):
    ax.axis('off')
    ax.set_title(cam_labels[cam], fontsize=11, fontweight='bold')
    im = ax.imshow(frms[0])
    txt = ax.text(5, 15, '', color='white', fontsize=10, fontweight='bold',
                  bbox=dict(facecolor='black', alpha=0.5, boxstyle='round,pad=0.2'))
    ims.append((im, frms))
    texts.append(txt)

fig.suptitle(f"Episode {MULTI_EPISODE_IDX} — All Cameras", fontsize=13, fontweight='bold')

def update_multi(i):
    artists = []
    for (im, frms), txt in zip(ims, texts):
        im.set_data(frms[i])
        txt.set_text(f"{i}/{min_frames-1}")
        artists.extend([im, txt])
    return artists

ani_multi = animation.FuncAnimation(fig, update_multi, frames=min_frames,
                                     interval=1000 // MULTI_FPS, blit=True)
plt.close()
HTML(ani_multi.to_jshtml())

## 7.5 Synchronized Video + Robot State Time-Series Viewer

카메라 영상과 robot state / action time-series를 **동일 타임라인** 위에서 함께 재생합니다.

- 왼쪽 : 선택한 카메라 영상 (프레임 단위 재생)
- 오른쪽 : Joint angles, Gripper, Base state — 빨간 **커서 선**이 현재 시각을 표시
- `head_rgb/time` 과 `samples/time` 의 타임스탬프로 **nearest-neighbor 동기화**

설정 변수
| 변수 | 설명 |
|---|---|
| `SYNC_EPISODE_IDX` | 에피소드 인덱스 |
| `SYNC_CAMERAS` | 보여줄 카메라 목록 (`head_rgb` / `left_rgb` / `right_rgb`) |
| `SYNC_FPS` | 재생 속도 (fps) |
| `SYNC_DEGREES` | `True` → 각도(°), `False` → radian |
| `SYNC_MAX_FRAMES` | 최대 프레임 수 (`None` = 전체) |

In [ ]:
# ================================================================
#  7.5  Synchronized Video + Robot State Time-Series Viewer
# ================================================================

# ── Animation embed size limit (MB) ──────────────────────────────
plt.rcParams['animation.embed_limit'] = 80   # raise from default 20 MB

# ── User options ─────────────────────────────────────────────────
SYNC_EPISODE_IDX = EPISODE_IDX              # episode index
SYNC_CAMERAS     = ['head_rgb']             # cameras to show
                                             # (add 'left_rgb'/'right_rgb' if needed)
SYNC_FPS         = 10                        # playback FPS
SYNC_DEGREES     = True                      # True → °, False → rad
SYNC_MAX_FRAMES  = None                      # int to limit frame count; None = all

# ── Camera display label ─────────────────────────────────────────
_CAM_LABEL = {
    'head_rgb':   'Head',
    'left_rgb':   'Left Wrist',
    'right_rgb':  'Right Wrist',
}

# ── Joint group metadata (matches JOINT_GROUPS in Section 5) ─────
_JG = [
    ("Torso",     range(0,  6),  "tab:blue"),
    ("Right Arm", range(6,  13), "tab:orange"),
    ("Left Arm",  range(13, 20), "tab:green"),
    ("Head",      range(20, 24), "tab:purple"),
]
def _jcol(j):
    for _, r, c in _JG:
        if j in r:
            return c
    return "gray"

# ── Load data ────────────────────────────────────────────────────
ep_dir_sv  = DATASET_ROOT / f"episode_{SYNC_EPISODE_IDX}"
h5_path_sv = list(ep_dir_sv.glob("*.h5"))[0]

with h5py.File(h5_path_sv, "r") as f:
    samp_time        = f["samples/time"][:]
    robot_pos_sv     = f["samples/robot_position"][:]        # (T_s, 24)
    robot_tgt_sv     = f["samples/robot_target_joints"][:]   # (T_s, 24)
    gripper_state_sv = f["samples/gripper_state"][:]         # (T_s,  2)
    gripper_tgt_sv   = f["samples/gripper_target"][:]        # (T_s,  2)
    base_state_sv    = f["samples/base_state"][:]            # (T_s,  3)

    cam_data = {}
    for cam in SYNC_CAMERAS:
        if f"{cam}/image" in f:
            t_key = f"{cam}/time"
            cam_data[cam] = {
                "frames": f[f"{cam}/image"][:],
                "time":   f[t_key][:] if t_key in f else None,
            }

valid_cams = [c for c in SYNC_CAMERAS if c in cam_data]
if not valid_cams:
    raise ValueError("No valid camera data found.")

# ── Time alignment (nearest-neighbour) ───────────────────────────
primary_cam = valid_cams[0]
primary_t   = cam_data[primary_cam]["time"]
n_cam_fr    = cam_data[primary_cam]["frames"].shape[0]
if SYNC_MAX_FRAMES is not None:
    n_cam_fr = min(n_cam_fr, int(SYNC_MAX_FRAMES))

samp_time_rel = samp_time - samp_time[0]
if primary_t is not None:
    cam_time_rel = (primary_t - samp_time[0])[:n_cam_fr]
else:
    cam_time_rel = np.linspace(0.0, samp_time_rel[-1], n_cam_fr)

def _nearest_idx(query, ref):
    idx = np.searchsorted(ref, query).clip(0, len(ref) - 1)
    idx_p = (idx - 1).clip(0, len(ref) - 1)
    closer = np.abs(ref[idx_p] - query) < np.abs(ref[idx] - query)
    idx[closer] = idx_p[closer]
    return idx

frame_to_si = _nearest_idx(cam_time_rel, samp_time_rel)

# ── Unit conversion ───────────────────────────────────────────────
_scale  = np.degrees if SYNC_DEGREES else (lambda x: x)
_unit   = "°" if SYNC_DEGREES else "rad"
rp_plot = _scale(robot_pos_sv)
rt_plot = _scale(robot_tgt_sv)
n_jt    = robot_pos_sv.shape[1]

# ================================================================
#  Build figure layout
#  Left  : cameras stacked (1 col per camera, span all rows)
#  Right : 3 state plots stacked
# ================================================================
n_cv  = len(valid_cams)
col_widths = [3] * n_cv + [5]
fig_w = sum(col_widths) * 1.35
fig_h = 9

fig_sv = plt.figure(figsize=(fig_w, fig_h))
gs = gridspec.GridSpec(
    3, n_cv + 1,
    figure=fig_sv,
    width_ratios=col_widths,
    hspace=0.50, wspace=0.30,
)

# Camera axes (each spans all 3 rows)
ax_cams = {}
for ci, cam in enumerate(valid_cams):
    ax_c = fig_sv.add_subplot(gs[:, ci])
    ax_c.axis("off")
    ax_c.set_title(_CAM_LABEL.get(cam, cam), fontsize=11, fontweight="bold", pad=6)
    ax_cams[cam] = ax_c

# State plot axes
ax_j = fig_sv.add_subplot(gs[0, n_cv])   # joint angles
ax_g = fig_sv.add_subplot(gs[1, n_cv])   # gripper
ax_b = fig_sv.add_subplot(gs[2, n_cv])   # base

T_end = samp_time_rel[-1]

# ── Static time-series ────────────────────────────────────────────
for j in range(n_jt):
    c = _jcol(j)
    ax_j.plot(samp_time_rel, rp_plot[:, j], color=c, lw=0.9, alpha=0.55)
    ax_j.plot(samp_time_rel, rt_plot[:, j], color=c, lw=0.7, alpha=0.28, ls="--")
ax_j.set_xlim(0, T_end)
ax_j.set_title(f"Joints ({_unit})  —  solid=State  dashed=Action", fontsize=9, fontweight="bold")
ax_j.set_ylabel(_unit, fontsize=8)
ax_j.grid(True, alpha=0.2)
leg_patches = [mpatches.Patch(color=c, label=g) for g, _, c in _JG]
ax_j.legend(handles=leg_patches, ncol=2, fontsize=7, loc="upper right", framealpha=0.7)

ax_g.plot(samp_time_rel, gripper_state_sv[:, 1], color="royalblue", lw=1.5,           label="L-State")
ax_g.plot(samp_time_rel, gripper_state_sv[:, 0], color="tomato",    lw=1.5,           label="R-State")
ax_g.plot(samp_time_rel, gripper_tgt_sv[:, 1],   color="royalblue", lw=1.2, ls="--", alpha=0.55, label="L-Action")
ax_g.plot(samp_time_rel, gripper_tgt_sv[:, 0],   color="tomato",    lw=1.2, ls="--", alpha=0.55, label="R-Action")
ax_g.set_xlim(0, T_end)
ax_g.set_title("Gripper  —  solid=State  dashed=Action", fontsize=9, fontweight="bold")
ax_g.set_ylabel("Value", fontsize=8)
ax_g.legend(ncol=2, fontsize=7, loc="upper right", framealpha=0.7)
ax_g.grid(True, alpha=0.2)

_bc = ["steelblue", "coral", "seagreen"]
_bl = ["x (m)", "y (m)", "yaw"]
for bi, (lbl, col) in enumerate(zip(_bl, _bc)):
    ax_b.plot(samp_time_rel, base_state_sv[:, bi], color=col, lw=1.5, label=lbl)
ax_b.set_xlim(0, T_end)
ax_b.set_title("Base State", fontsize=9, fontweight="bold")
ax_b.set_ylabel("Value", fontsize=8)
ax_b.set_xlabel("Time (s)", fontsize=8)
ax_b.legend(ncol=3, fontsize=7, framealpha=0.7)
ax_b.grid(True, alpha=0.2)

fig_sv.suptitle(
    f"Episode {SYNC_EPISODE_IDX}  —  Synchronized Video + Robot State",
    fontsize=13, fontweight="bold", y=1.01,
)

# ── Dynamic artists ───────────────────────────────────────────────
cam_ims, cam_txts = {}, {}
for cam, ax_c in ax_cams.items():
    im  = ax_c.imshow(cam_data[cam]["frames"][0])
    txt = ax_c.text(
        5, 18, "", color="white", fontsize=9, fontweight="bold",
        bbox=dict(facecolor="black", alpha=0.55, boxstyle="round,pad=0.2"),
    )
    cam_ims[cam]  = im
    cam_txts[cam] = txt

# Red cursor lines
cur_j = ax_j.axvline(0.0, color="red", lw=1.6, ls="--", alpha=0.85, zorder=5)
cur_g = ax_g.axvline(0.0, color="red", lw=1.6, ls="--", alpha=0.85, zorder=5)
cur_b = ax_b.axvline(0.0, color="red", lw=1.6, ls="--", alpha=0.85, zorder=5)

# Moving dots on gripper (state)
dot_gl, = ax_g.plot([], [], "o", color="royalblue", ms=7, zorder=6)
dot_gr, = ax_g.plot([], [], "o", color="tomato",    ms=7, zorder=6)

# Moving dots on joint plot (group mean of state)
grp_dots = {}
for grp, rng, col in _JG:
    dot, = ax_j.plot([], [], "o", color=col, ms=5, zorder=6, alpha=0.85)
    grp_dots[grp] = (dot, list(rng))

# ── Animation update ──────────────────────────────────────────────
def _update_sync(fi):
    si = int(frame_to_si[fi])
    t  = samp_time_rel[si]
    artists = []

    # Camera frames
    for cam in valid_cams:
        cam_ims[cam].set_data(cam_data[cam]["frames"][fi])
        cam_txts[cam].set_text(f"f={fi}  t={t:.2f}s")
        artists += [cam_ims[cam], cam_txts[cam]]

    # Cursors
    for cur in [cur_j, cur_g, cur_b]:
        cur.set_xdata([t, t])
        artists.append(cur)

    # Gripper dots
    dot_gl.set_data([t], [gripper_state_sv[si, 1]])
    dot_gr.set_data([t], [gripper_state_sv[si, 0]])
    artists += [dot_gl, dot_gr]

    # Joint group mean dots
    for grp, (dot, idxs) in grp_dots.items():
        dot.set_data([t], [rp_plot[si, idxs].mean()])
        artists.append(dot)

    return artists

ani_sv = animation.FuncAnimation(
    fig_sv, _update_sync,
    frames=n_cam_fr,
    interval=1000 // SYNC_FPS,
    blit=True,
)
plt.close()

print(f"Episode {SYNC_EPISODE_IDX} | cameras={valid_cams} | "
      f"frames={n_cam_fr} | sample steps={len(samp_time_rel)} | "
      f"duration={T_end:.2f}s | FPS={SYNC_FPS}")
print("※ 카메라 추가: SYNC_CAMERAS = ['head_rgb', 'left_rgb', 'right_rgb']")

HTML(ani_sv.to_jshtml())


## 8. Visualize Training-Transformed Images (same preprocessing path as training)

In [ ]:
import sys
# NOTE:
# This cell reproduces the training-time image preprocessing path as closely as possible for RBY1 (pi05_rby1):
#   1) Rby1Inputs (policy adapter) -> unified image keys
#   2) ResizeImages(224,224) (model transform)
#   3) Observation.from_dict() (uint8 -> [-1,1])
#   4) preprocess_observation(train=True) (random crop/resize/rotate + color jitter)
# and visualizes RAW vs TRANSFORMED frames as a video.

from pathlib import Path

if 'cam_labels' not in globals():
    cam_labels = {'head_rgb': 'Head Camera', 'left_rgb': 'Left Camera', 'right_rgb': 'Right Camera'}

if 'DATASET_ROOT' not in globals():
    DATASET_ROOT = Path('/media/hyunjin/T7/rby1_demo/PuttingCupintotheDishV2')

# ----------------------------
# User options
# ----------------------------
TRAIN_VIS_EPISODE_IDX = 0
TRAIN_VIS_FPS = 10
TRAIN_VIS_MAX_FRAMES = 300     # limit for quick preview; set None to use all
TRAIN_VIS_SEED = 0             # augmentation RNG seed (change to view another random augmentation stream)
CAMS = ['head_rgb', 'left_rgb', 'right_rgb']

# ----------------------------
# Imports from openpi (repo-local)
# ----------------------------
repo_root = Path('/home/hyunjin/rby1_ws/openpi')  # absolute path to openpi repo
if str(repo_root / 'src') not in sys.path:
    sys.path.append(str(repo_root / 'src'))

try:
    import jax
    from openpi import transforms as opi_transforms
    from openpi.models import model as opi_model
    from openpi.models import pi0_config as opi_pi0_config
    from openpi.policies import rby1_policy
except Exception as e:
    raise RuntimeError(
        "Failed to import openpi/jax modules for training-transform visualization. "
        "Please run this notebook in the openpi environment.\n"
        f"Original error: {e}"
    )

# ----------------------------
# Load episode data
# ----------------------------
ep_dir_t = DATASET_ROOT / f"episode_{TRAIN_VIS_EPISODE_IDX}"
h5_candidates_t = list(ep_dir_t.glob("*.h5"))
if not h5_candidates_t:
    raise FileNotFoundError(f"No .h5 found under {ep_dir_t}")
h5_path_t = h5_candidates_t[0]

with h5py.File(h5_path_t, 'r') as f:
    raw_cam_frames = {}
    for cam in CAMS:
        key = f"{cam}/image"
        raw_cam_frames[cam] = f[key][:] if key in f else None

    # Try to use actual robot state; fallback to zeros if absent.
    if 'samples/robot_position' in f:
        state_seq = f['samples/robot_position'][:]
    elif 'samples/state' in f:
        state_seq = f['samples/state'][:]
    else:
        state_seq = None

available_cams = [c for c in CAMS if raw_cam_frames[c] is not None]
if not available_cams:
    raise ValueError(f"No camera images found among {CAMS} in {h5_path_t}")

# Build length constraints
length_candidates = [raw_cam_frames[c].shape[0] for c in available_cams]
if state_seq is not None:
    length_candidates.append(state_seq.shape[0])
num_frames = min(length_candidates)
if TRAIN_VIS_MAX_FRAMES is not None:
    num_frames = min(num_frames, int(TRAIN_VIS_MAX_FRAMES))

# ----------------------------
# Prepare transforms (same path as training config for pi05_rby1)
# ----------------------------
model_cfg = opi_pi0_config.Pi0Config(pi05=True)
data_tf = rby1_policy.Rby1Inputs(model_type=model_cfg.model_type)
resize_tf = opi_transforms.ResizeImages(224, 224)

# Mapping from dataset camera key -> RBY1 policy input key
to_obs_key = {
    'head_rgb': 'observation/head_image',
    'left_rgb': 'observation/left_wrist_image',
    'right_rgb': 'observation/right_wrist_image',
}

# For missing camera streams, use zeros from first available camera shape.
fallback_shape = raw_cam_frames[available_cams[0]][0].shape
zero_img = np.zeros(fallback_shape, dtype=np.uint8)

def get_state_at(i):
    if state_seq is None:
        return np.zeros(16, dtype=np.float32)
    s = np.asarray(state_seq[i]).astype(np.float32).reshape(-1)
    # Rby1Inputs expects 1D state; training later pads/slices as needed.
    if s.shape[0] >= 16:
        return s[:16]
    out = np.zeros(16, dtype=np.float32)
    out[: s.shape[0]] = s
    return out

def to_uint8_from_model_image(img_float_or_uint8):
    arr = np.asarray(img_float_or_uint8)
    if arr.dtype == np.uint8:
        return arr
    # Model observation images are in [-1, 1].
    arr = np.clip((arr + 1.0) * 0.5, 0.0, 1.0)
    return (arr * 255.0).astype(np.uint8)

# Precompute transformed frames (deterministic sequence from TRAIN_VIS_SEED)
rng = jax.random.key(TRAIN_VIS_SEED)
processed = {cam: [] for cam in available_cams}

for i in range(num_frames):
    sample = {
        'observation/state': get_state_at(i),
        'prompt': 'visualize training preprocessing',
    }
    for cam in CAMS:
        frame = raw_cam_frames[cam][i] if raw_cam_frames[cam] is not None else zero_img
        sample[to_obs_key[cam]] = frame

    # 1) policy adapter -> {'image', 'image_mask', 'state', ...}
    x = data_tf(sample)
    # 2) model transform resize (with padding)
    x = resize_tf(x)

    # Add batch dimension for model preprocessing
    batched = {
        'image': {k: v[None, ...] for k, v in x['image'].items()},
        'image_mask': {k: np.asarray([m], dtype=bool) for k, m in x['image_mask'].items()},
        'state': x['state'][None, ...],
    }

    # 3) uint8 -> [-1,1], 4) train-time augmentations
    obs = opi_model.Observation.from_dict(batched)
    rng, sub = jax.random.split(rng)
    obs_aug = opi_model.preprocess_observation(sub, obs, train=True, image_keys=list(obs.images.keys()))

    # Collect camera outputs using model image keys
    out_key = {
        'head_rgb': 'base_0_rgb',
        'left_rgb': 'left_wrist_0_rgb',
        'right_rgb': 'right_wrist_0_rgb',
    }
    for cam in available_cams:
        img_aug = np.asarray(obs_aug.images[out_key[cam]][0])
        processed[cam].append(to_uint8_from_model_image(img_aug))

print(f"Episode {TRAIN_VIS_EPISODE_IDX} | frames={num_frames} | cams={available_cams}")
print("Top row: raw display, Bottom row: transformed (training-time preprocessing)")

# ----------------------------
# Build animation: top=raw, bottom=transformed
# ----------------------------
n_cams = len(available_cams)
fig, axes = plt.subplots(2, n_cams, figsize=(5 * n_cams, 6))
if n_cams == 1:
    axes = np.array(axes).reshape(2, 1)

img_artists = []
text_artists = []

for j, cam in enumerate(available_cams):
    raw0 = raw_cam_frames[cam][0]
    aug0 = processed[cam][0]

    # Row 0: raw
    ax_raw = axes[0, j]
    ax_raw.axis('off')
    ax_raw.set_title(f"{cam_labels.get(cam, cam)} | RAW", fontsize=11, fontweight='bold')
    im_raw = ax_raw.imshow(raw0)
    txt_raw = ax_raw.text(
        5, 15, '', color='white', fontsize=9, fontweight='bold',
        bbox=dict(facecolor='black', alpha=0.5, boxstyle='round,pad=0.2')
    )

    # Row 1: transformed
    ax_aug = axes[1, j]
    ax_aug.axis('off')
    ax_aug.set_title(f"{cam_labels.get(cam, cam)} | TRAIN-TRANSFORMED", fontsize=11, fontweight='bold')
    im_aug = ax_aug.imshow(aug0)
    txt_aug = ax_aug.text(
        5, 15, '', color='white', fontsize=9, fontweight='bold',
        bbox=dict(facecolor='black', alpha=0.5, boxstyle='round,pad=0.2')
    )

    img_artists.append((cam, im_raw, im_aug))
    text_artists.append((txt_raw, txt_aug))

fig.suptitle(
    f"Episode {TRAIN_VIS_EPISODE_IDX} — Raw vs Training-Transformed Frames (seed={TRAIN_VIS_SEED})",
    fontsize=13, fontweight='bold'
)

def update_train_vis(i):
    artists = []
    for (cam, im_raw, im_aug), (txt_raw, txt_aug) in zip(img_artists, text_artists):
        im_raw.set_data(raw_cam_frames[cam][i])
        im_aug.set_data(processed[cam][i])
        txt_raw.set_text(f"raw {i}/{num_frames-1}")
        txt_aug.set_text(f"aug {i}/{num_frames-1}")
        artists.extend([im_raw, im_aug, txt_raw, txt_aug])
    return artists

ani_train = animation.FuncAnimation(
    fig, update_train_vis, frames=num_frames, interval=1000 // TRAIN_VIS_FPS, blit=True
)
plt.close()
HTML(ani_train.to_jshtml())

In [ ]:
from pathlib import Path

try:
    import imageio.v2 as imageio
except ImportError:
    raise ImportError("imageio is required. Install with: pip install imageio imageio-ffmpeg")

# ----------------------------
# User options  (필요에 따라 수정)
# ----------------------------
DEMO_ROOT   = Path("/media/hyunjin/T7/rby1_demo/PuttingCupintotheDishV2")  # 데모 루트 경로
EPISODE_IDX = 5       # 저장할 에피소드 인덱스
OUTPUT_DIR  = Path("./saved_videos")                                      # MP4 저장 폴더
FPS         = 10      # 출력 영상 FPS
CAMERAS     = ["head_rgb", "left_rgb", "right_rgb"]                       # 저장할 카메라

# ----------------------------
# 파일 확인
# ----------------------------
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ep_dir   = DEMO_ROOT / f"episode_{EPISODE_IDX}"
h5_files = sorted(ep_dir.glob("*.h5"))
if not h5_files:
    raise FileNotFoundError(f"No .h5 file found in {ep_dir}")
h5_path = h5_files[0]
print(f"Using: {h5_path}")



# ----------------------------
# MP4 저장
# ----------------------------
saved = []
with h5py.File(h5_path, "r") as f:
    for cam in CAMERAS:
        key = f"{cam}/image"
        if key not in f:
            print(f"[skip] {cam}: '{key}' not found in HDF5")
            continue

        frames = f[key][:]  # (T, H, W, 3)
        if frames.ndim != 4 or frames.shape[-1] != 3:
            print(f"[skip] {cam}: unexpected shape {frames.shape}")
            continue

        frames = frames.astype(np.uint8)
        out_path = OUTPUT_DIR / f"episode_{EPISODE_IDX}_{cam}.mp4"

        with imageio.get_writer(str(out_path), fps=FPS) as writer:
            for i in range(frames.shape[0]):
                writer.append_data(frames[i])

        saved.append(out_path)
        print(f"[ok] {cam}: {out_path}  "
              f"(frames={frames.shape[0]}, {frames.shape[2]}x{frames.shape[1]}, fps={FPS})")

print("\n=== Saved files ===")
for p in saved:
    print(f"  {p}")

## 9. Save Composite Video — Head RGB + Arm/Gripper Time-Series (MP4)

Head RGB 카메라 영상과 Right Arm / Left Arm / Left Gripper / Right Gripper time-series를  
한 화면에 합성하여 MP4로 저장합니다.

| 설정 변수 | 설명 |
|---|---|
| `COMP_EPISODE_IDX` | 에피소드 인덱스 (기본값: 전역 `EPISODE_IDX`) |
| `COMP_OUTPUT_DIR`  | 저장 폴더 |
| `COMP_FPS`         | 출력 영상 FPS |
| `COMP_MAX_FRAMES`  | 최대 프레임 수 (`None` = 전체) |
| `COMP_DPI`         | 렌더링 해상도 (높을수록 느림) |


In [ ]:
import matplotlib.backends.backend_agg as agg
from pathlib import Path

try:
    import imageio.v2 as imageio
except ImportError:
    raise ImportError("imageio is required: pip install imageio imageio-ffmpeg")

# ── User options ──────────────────────────────────────────────────
COMP_EPISODE_IDX = EPISODE_IDX  # 저장할 에피소드 인덱스 (전역 EPISODE_IDX 기본값 사용)
COMP_OUTPUT_DIR  = Path("./saved_videos")
COMP_FPS         = 15            # 출력 FPS
COMP_MAX_FRAMES  = None          # None = 전체 프레임
COMP_DPI         = 100           # 렌더링 해상도 (낮을수록 빠름)
COMP_DEGREES     = True          # True → °, False → rad

# H5 경로 결정: COMP_EPISODE_IDX로 직접 조회
_comp_ep_dir   = DATASET_ROOT / f"episode_{COMP_EPISODE_IDX}"
_comp_h5_list  = sorted(_comp_ep_dir.glob("*.h5"))
if not _comp_h5_list:
    raise FileNotFoundError(f"No .h5 file found in {_comp_ep_dir}")
COMP_H5_PATH = Path(_comp_h5_list[0])

# episode index label  (파일 경로에서 추출)
_ep_label = f"episode_{COMP_EPISODE_IDX}"

# ── Joint index mapping (consistent with JOINT_GROUPS in Section 5) ──
RIGHT_IDXS = list(range(6,  13))    # J6  – J12  (Right Arm)
LEFT_IDXS  = list(range(13, 20))    # J13 – J19  (Left Arm)

with h5py.File(COMP_H5_PATH, "r") as f:
    samp_time    = f["samples/time"][:]
    robot_pos_c  = f["samples/robot_position"][:]        # (T, 24)
    robot_tgt_c  = f["samples/robot_target_joints"][:]   # (T, 24)
    grp_state_c  = f["samples/gripper_state"][:]         # (T,  2)
    grp_tgt_c    = f["samples/gripper_target"][:]        # (T,  2)
    cam_frames_c = f["head_rgb/image"][:]                # (T, H, W, 3)
    cam_time_c   = f["head_rgb/time"][:] if "head_rgb/time" in f else None


n_cam = cam_frames_c.shape[0]
if COMP_MAX_FRAMES is not None:
    n_cam = min(n_cam, int(COMP_MAX_FRAMES))

# Nearest-neighbor timestamp alignment
samp_rel = samp_time - samp_time[0]
cam_rel  = (cam_time_c - samp_time[0])[:n_cam] if cam_time_c is not None \
           else np.linspace(0, samp_rel[-1], n_cam)
T_end    = samp_rel[-1]

def _nn_idx(query, ref):
    idx   = np.searchsorted(ref, query).clip(0, len(ref) - 1)
    idx_p = (idx - 1).clip(0, len(ref) - 1)
    closer = np.abs(ref[idx_p] - query) < np.abs(ref[idx] - query)
    idx[closer] = idx_p[closer]
    return idx

frame_to_si = _nn_idx(cam_rel, samp_rel)

# Unit conversion
_sc = np.degrees if COMP_DEGREES else (lambda x: x)
_u  = "°" if COMP_DEGREES else "rad"
rp  = _sc(robot_pos_c)
rt  = _sc(robot_tgt_c)

# ── Build figure layout ───────────────────────────────────────────
# Col 0 (spans 2 rows) : Head RGB
# Col 1, Row 0 : Right Arm   |  Col 1, Row 1 : Left Arm
# Col 2, Row 0 : Left Gripper|  Col 2, Row 1 : Right Gripper
fig = plt.figure(figsize=(18, 8), dpi=COMP_DPI)
gs  = gridspec.GridSpec(2, 3, figure=fig,
                        width_ratios=[2, 3, 2],
                        hspace=0.50, wspace=0.38)

ax_cam = fig.add_subplot(gs[:, 0])
ax_ra  = fig.add_subplot(gs[0, 1])
ax_la  = fig.add_subplot(gs[1, 1])
ax_lg  = fig.add_subplot(gs[0, 2])
ax_rg  = fig.add_subplot(gs[1, 2])

ax_cam.axis("off")

# ── Static time-series ────────────────────────────────────────────
for j in RIGHT_IDXS:
    ax_ra.plot(samp_rel, rp[:, j], color="tab:orange", lw=0.9, alpha=0.65)
    ax_ra.plot(samp_rel, rt[:, j], color="tab:orange", lw=0.7, alpha=0.28, ls="--")
ax_ra.set_title(f"Right Arm ({_u})  solid=State  dash=Action", fontsize=9, fontweight="bold")
ax_ra.set_ylabel(_u, fontsize=8); ax_ra.set_xlim(0, T_end); ax_ra.grid(True, alpha=0.2)

for j in LEFT_IDXS:
    ax_la.plot(samp_rel, rp[:, j], color="tab:green", lw=0.9, alpha=0.65)
    ax_la.plot(samp_rel, rt[:, j], color="tab:green", lw=0.7, alpha=0.28, ls="--")
ax_la.set_title(f"Left Arm ({_u})  solid=State  dash=Action", fontsize=9, fontweight="bold")
ax_la.set_ylabel(_u, fontsize=8); ax_la.set_xlabel("Time (s)", fontsize=8)
ax_la.set_xlim(0, T_end); ax_la.grid(True, alpha=0.2)

ax_lg.plot(samp_rel, grp_state_c[:, 1], color="royalblue", lw=1.5, label="State")
ax_lg.plot(samp_rel, grp_tgt_c[:,   1], color="royalblue", lw=1.2, ls="--", alpha=0.6, label="Action")
ax_lg.fill_between(samp_rel, grp_state_c[:, 1], grp_tgt_c[:, 1], alpha=0.12, color="royalblue")
ax_lg.set_title("Left Gripper", fontsize=9, fontweight="bold")
ax_lg.set_ylabel("Value", fontsize=8); ax_lg.legend(fontsize=7)
ax_lg.set_xlim(0, T_end); ax_lg.grid(True, alpha=0.2)

ax_rg.plot(samp_rel, grp_state_c[:, 0], color="tomato", lw=1.5, label="State")
ax_rg.plot(samp_rel, grp_tgt_c[:,   0], color="tomato", lw=1.2, ls="--", alpha=0.6, label="Action")
ax_rg.fill_between(samp_rel, grp_state_c[:, 0], grp_tgt_c[:, 0], alpha=0.12, color="tomato")
ax_rg.set_title("Right Gripper", fontsize=9, fontweight="bold")
ax_rg.set_ylabel("Value", fontsize=8); ax_rg.set_xlabel("Time (s)", fontsize=8)
ax_rg.legend(fontsize=7); ax_rg.set_xlim(0, T_end); ax_rg.grid(True, alpha=0.2)

fig.suptitle(
    f"{_ep_label}  —  Head RGB + Right/Left Arm + Gripper",
    fontsize=13, fontweight="bold"
)

# ── Dynamic artists ────────────────────────────────────────────────
im_cam   = ax_cam.imshow(cam_frames_c[0])
cur_ra   = ax_ra.axvline(0, color="red", lw=1.6, ls="--", alpha=0.9, zorder=5)
cur_la   = ax_la.axvline(0, color="red", lw=1.6, ls="--", alpha=0.9, zorder=5)
cur_lg   = ax_lg.axvline(0, color="red", lw=1.6, ls="--", alpha=0.9, zorder=5)
cur_rg   = ax_rg.axvline(0, color="red", lw=1.6, ls="--", alpha=0.9, zorder=5)
dot_ra, = ax_ra.plot([], [], "o", color="darkorange", ms=6, zorder=6)
dot_la, = ax_la.plot([], [], "o", color="seagreen",   ms=6, zorder=6)
dot_lg, = ax_lg.plot([], [], "o", color="royalblue",  ms=7, zorder=6)
dot_rg, = ax_rg.plot([], [], "o", color="tomato",     ms=7, zorder=6)

# ── Render and save ────────────────────────────────────────────────
canvas   = agg.FigureCanvasAgg(fig)
out_path = COMP_OUTPUT_DIR / f"{_ep_label}_composite.mp4"

print(f"Saving {n_cam} frames → {out_path}  (FPS={COMP_FPS}, DPI={COMP_DPI})")
print("Progress: ", end="", flush=True)

with imageio.get_writer(str(out_path), fps=COMP_FPS, macro_block_size=1) as writer:
    for fi in range(n_cam):
        si = int(frame_to_si[fi])
        t  = float(samp_rel[si])

        im_cam.set_data(cam_frames_c[fi])
        ax_cam.set_title(f"Head RGB  f={fi}/{n_cam-1}  t={t:.2f}s",
                         fontsize=10, fontweight="bold")
        for cur in [cur_ra, cur_la, cur_lg, cur_rg]:
            cur.set_xdata([t, t])
        dot_ra.set_data([t], [rp[si, RIGHT_IDXS].mean()])
        dot_la.set_data([t], [rp[si, LEFT_IDXS].mean()])
        dot_lg.set_data([t], [grp_state_c[si, 1]])
        dot_rg.set_data([t], [grp_state_c[si, 0]])

        canvas.draw()
        buf = np.asarray(canvas.buffer_rgba())[:, :, :3]   # RGBA → RGB
        writer.append_data(buf)

        if fi % max(1, n_cam // 20) == 0:
            print(f"{fi/n_cam*100:.0f}%.. ", end="", flush=True)

plt.close(fig)
print(f"\n✅ Done: {out_path}")
print(f"   episode={_ep_label}, frames={n_cam}, "
      f"duration={n_cam/COMP_FPS:.1f}s, size={out_path.stat().st_size/1024:.0f} KB")


## 10. Bulk GIF Export — All Episodes, All Cameras

전체 에피소드의 모든 카메라 영상을 GIF 파일로 일괄 저장합니다.  
GIF는 각 에피소드 폴더(H5 파일 옆)에 저장되며, 이미 파일이 있으면 스킵합니다.

| 설정 변수 | 설명 |
|---|---|
| `GIF_CAMERAS` | 내보낼 카메라 목록 (`head` / `right` / `left`) |
| `GIF_FPS` | GIF 재생 속도 (fps) |
| `GIF_RESIZE` | 리사이즈 크기 `(width, height)`; `None` = 원본 해상도 |
| `GIF_OVERWRITE` | `True` = 기존 GIF 덮어쓰기 |

In [ ]:
try:
    from PIL import Image as _PIL_Image
except ImportError:
    raise ImportError("Pillow is required: pip install Pillow")

# ── User options ───────────────────────────────────────────────────
GIF_CAMERAS   = ['head', 'right', 'left']  # camera prefixes  (key: {cam}_rgb/image)
GIF_FPS       = 30                          # GIF playback FPS
GIF_RESIZE    = (320, 240)                  # (width, height); None = keep original
GIF_OVERWRITE = False                       # True = overwrite existing GIFs


def export_episode_gif(h5_path: Path, camera: str, fps: int = 30,
                       resize=None, overwrite: bool = False) -> bool:
    """Export a GIF for one camera from an HDF5 episode file.

    The GIF is saved alongside the H5 file.
    Returns True on success, False on skip/error.
    """
    key = f"{camera}_rgb/image"
    out_path = h5_path.parent / f"{h5_path.stem}_{camera}.gif"

    if not overwrite and out_path.exists():
        print(f"  ⏭️  {camera}: already exists — skip")
        return True

    try:
        with h5py.File(h5_path, 'r') as f:
            if key not in f:
                print(f"  ⚠️  {camera}: key '{key}' not found in HDF5 — skip")
                return False
            frames = f[key][:]   # (T, H, W, 3)
    except Exception as e:
        print(f"  ❌ {camera}: read error — {e}")
        return False

    if frames.ndim != 4 or frames.shape[-1] != 3:
        print(f"  ❌ {camera}: unexpected shape {frames.shape} — skip")
        return False


    # Build PIL frames
    pil_frames = []
    for frame in frames:
        img = _PIL_Image.fromarray(frame.astype(np.uint8), 'RGB')
        if resize is not None:
            img = img.resize(resize, _PIL_Image.LANCZOS)
        pil_frames.append(img)

    # Save GIF
    duration_ms = int(1000 / fps)
    pil_frames[0].save(
        out_path, save_all=True, append_images=pil_frames[1:],
        duration=duration_ms, loop=0,
    )
    w, h = pil_frames[0].size
    print(f"  ✅ {camera}: {out_path.name}  ({w}x{h}, {len(pil_frames)} frames, {fps}fps)")
    return True


# ── Main loop: iterate over all episodes ───────────────────────────
ep_dirs = sorted(DATASET_ROOT.glob("episode_*"),
                 key=lambda x: int(x.name.split("_")[1]))

print(f"🎬 {len(ep_dirs)} episodes found in: {DATASET_ROOT}")
print(f"   cameras={GIF_CAMERAS}, fps={GIF_FPS}, resize={GIF_RESIZE}, overwrite={GIF_OVERWRITE}\n")

success, skipped, failed = 0, 0, 0
for ep_dir in ep_dirs:
    h5_candidates = sorted(ep_dir.glob("*.h5"))
    if not h5_candidates:
        print(f"[{ep_dir.name}] ⚠️  no .h5 file found — skip")
        skipped += len(GIF_CAMERAS)
        continue

    h5_path = h5_candidates[0]
    print(f"\n{'─'*52}\n[{ep_dir.name}]  {h5_path.name}")
    for cam in GIF_CAMERAS:
        result = export_episode_gif(
            h5_path, cam,
            fps=GIF_FPS, resize=GIF_RESIZE, overwrite=GIF_OVERWRITE,
        )
        if result:
            success += 1
        else:
            failed += 1

print(f"\n{'═'*52}")
print(f"✅ Bulk GIF export complete!")
print(f"   success={success}, skipped(no-h5)={skipped}, failed={failed}")
print(f"{'═'*52}")